## Getting the Data


In [31]:
import pandas as pd
import re

In [32]:
messages = pd.read_csv(
    "data/SMSSpamCollection", delimiter="\t", names=["label", "message"]
)

In [ ]:
messages.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


: 

## Data Cleaning and Preprocessing


In [33]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [34]:
wordnetlemmatizer = WordNetLemmatizer()

In [35]:
corpus = []

for i in range(len(messages)):
    message = re.sub("[^a-zA-Z]", " ", messages["message"][i])
    message = message.lower()
    message = message.split()

    message = [
        wordnetlemmatizer.lemmatize(word)
        for word in message
        if word not in set(stopwords.words("english"))
    ]

    message = " ".join(message)

    corpus.append(message)

## Encoding Target Variable (y)


In [36]:
# Encoding dependent variable: ham -> 0, spam -> 1
y = pd.get_dummies(messages["label"], drop_first=True)
y = y.iloc[:, 0].values

## Train Test Split


In [37]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    corpus, y, test_size=0.20, random_state=0
)

## Apply TF-IDF


In [38]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [39]:
tfidf = TfidfVectorizer(max_features=2500, ngram_range=(1, 2))

X_train = tfidf.fit_transform(X_train).toarray()
X_test = tfidf.transform(X_test).toarray()

## Training Model using Naive Bayes Classifier


In [40]:
from sklearn.naive_bayes import MultinomialNB

spam_detect_model = MultinomialNB().fit(X_train, y_train)

## Model Prediction


In [41]:
y_pred = spam_detect_model.predict(X_test)

## Model Evaluation


In [42]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [43]:
confusion_m = confusion_matrix(y_test, y_pred)
confusion_m

array([[954,   1],
       [ 24, 136]])

In [44]:
accuracy = accuracy_score(y_test, y_pred)
accuracy

0.9775784753363229

In [45]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

       False       0.98      1.00      0.99       955
        True       0.99      0.85      0.92       160

    accuracy                           0.98      1115
   macro avg       0.98      0.92      0.95      1115
weighted avg       0.98      0.98      0.98      1115

